In [49]:
import pypsa
import pandas as pd
import numpy as np

In [50]:
# df_electric_load erstellen (LED-Beleuchtung mit 16h an, 500 W, und 8h aus, 0W)

from datetime import datetime, timedelta
# Startdatum
start_datetime = datetime(2025, 1, 1, 0, 0)
total_hours = 365 * 24

timestamps = []
power_values = []

current_time = start_datetime

for day in range(365):
    for hour in range(16):
        timestamps.append(current_time)
        power_values.append(500)
        current_time += timedelta(hours=1)

    for hour in range(8):
        timestamps.append(current_time)
        power_values.append(0)
        current_time += timedelta(hours=1)


df_electric_load = pd.DataFrame({
    'Timestamp': timestamps,
    'Power_W': power_values
})

In [51]:
# Parameter
pv_costs = 1200 # €/kWp
heat_storage_loss = 0.005 #decimal

In [54]:
# Netzwerk erstellen
network = pypsa.Network()
network.set_snapshots(range(8760))

#Buses
network.add('Bus', name = 'electricity')
network.add('Bus', name = 'heat')
network.add('Bus', name = 'water')


#Carriers

#Generators
network.add('Generator', name = 'pv', bus = 'electricity',
           marginal_cost = pv_costs, p_nom = pv_p_nom)

#Links
network.add('Link', name = 'heat_pump', bus0 = 'electricity',
            bus1='heat', p_nom_extendable = True)
network.add('Link', name = 'water_pump', bus0 = 'electricity',
            bus1='water',  p_nom_extendable = True)


#Stores
network.add('Store', name = 'battery', bus = 'electricity',
           e_nom_extendable = True, e_cyclic = True)
network.add('Store', name = 'heat_store', bus = 'heat',
           e_nom_extendable = True,
           standing_loss = heat_storage_loss, e_cyclic = True)
network.add('Store', name = 'water_store', bus = 'water',
           p_nom_extendable = True, e_cyclic = True)

#Loads
network.add('Load', name = 'electrical_load', bus = 'electricity', p_set = list(df_electric_load['Power_W']), p_nom = pv_p_nom)

# network.add('Load', name = 'thermal_load', bus = 'heat', p_set = heat_data)
# network.add('Load', name = 'water_load', bus = 'water', p_set = water_data)


Index(['electrical_load'], dtype='object')